In [46]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"

Instead of single file the entire folder

In [121]:
import json
import re
import numpy as np
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Any

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# -------------------------
# CONFIG
# -------------------------
MODEL_DIR = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/siamese_sentence_ttp_model_bosch"
TTP_DESC_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/ttp_descriptions2.json"

INPUT_DIR = Path("/home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt")          # <- folder with many .json files
OUTPUT_DIR = Path("/home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps")  # <- new folder to write outputs

RECURSIVE = False  # set True if json files are in nested subfolders

SIM_THRESHOLD = 0.3
MAX_SENT_LEN = 500

MAX_LEN_SENT = 192
MAX_LEN_TTP = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# SENTENCE SPLITTING / CLEANING
# -------------------------
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")
TTP_REGEX = re.compile(r"\bT\d{4}(?:\.\d{3})?\b")


def remove_ttp_ids(text: str) -> str:
    cleaned = TTP_REGEX.sub("", text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def split_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text)
    return [
        s.strip()
        for s in _SENT_SPLIT.split(text)
        if 10 <= len(s.strip()) <= MAX_SENT_LEN
    ]


def get_subtechniques(parent_id: str, ttp_desc: Dict[str, str]) -> List[str]:
    prefix = parent_id + "."
    return [k for k in ttp_desc.keys() if k.startswith(prefix)]


# -------------------------
# HF ENCODER (mean pool + normalize)
# -------------------------
@torch.no_grad()
def encode_texts(
    model: AutoModel,
    tokenizer: AutoTokenizer,
    texts: List[str],
    max_length: int,
    batch_size: int = 64,
) -> np.ndarray:
    if not texts:
        return np.zeros((0, model.config.hidden_size), dtype=np.float32)

    model.eval()
    all_vecs = []

    for i in range(0, len(texts), batch_size):
        chunk = texts[i : i + batch_size]
        tok = tokenizer(
            chunk,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        out = model(**tok)
        last_hidden = out.last_hidden_state
        mask = tok["attention_mask"].unsqueeze(-1).type_as(last_hidden)

        pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        pooled = F.normalize(pooled, p=2, dim=1)

        all_vecs.append(pooled.detach().cpu().numpy())

    return np.vstack(all_vecs)


def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return a @ b.T


# -------------------------
# PER-DOC PROCESSING
# -------------------------
def process_doc(
    doc: dict,
    model: AutoModel,
    tokenizer: AutoTokenizer,
    ttp_desc: Dict[str, str],
) -> dict:
    if "original_txt" not in doc or "ID_list" not in doc:
        doc["TTP_sentences"] = {}
        doc["_warning"] = "Missing required keys: 'original_txt' and/or 'ID_list'"
        return doc

    text = remove_ttp_ids(doc["original_txt"])
    candidate_ttps = doc["ID_list"]

    ttp_ids = [t for t in candidate_ttps if t in ttp_desc]
    ttp_texts = [f"{t}: {ttp_desc[t]}" for t in ttp_ids]

    doc["TTP_sentences"] = {}
    if not ttp_ids:
        return doc

    ttp_embs = encode_texts(model, tokenizer, ttp_texts, MAX_LEN_TTP, batch_size=32)

    sentences = split_sentences(text)
    if not sentences:
        return doc

    sent_embs = encode_texts(model, tokenizer, sentences, MAX_LEN_SENT, batch_size=64)
    sims = cosine_sim_matrix(sent_embs, ttp_embs)

    ttp_sentence_map = defaultdict(list)
    for i, sent in enumerate(sentences):
        good = np.where(sims[i] >= SIM_THRESHOLD)[0]
        for j in good:
            ttp_id = ttp_ids[j]
            score = float(sims[i, j])
            ttp_sentence_map[ttp_id].append({"sentence": sent, "score": round(score, 4)})

    doc["TTP_sentences"] = dict(ttp_sentence_map)

    missing_ttps = [
        t
        for t in ttp_ids
        if t not in doc["TTP_sentences"] or len(doc["TTP_sentences"][t]) == 0
    ]

    for parent in missing_ttps:
        if "." in parent:
            continue

        subs = get_subtechniques(parent, ttp_desc)
        if not subs:
            continue

        sub_texts = [f"{t}: {ttp_desc[t]}" for t in subs]
        sub_embs = encode_texts(model, tokenizer, sub_texts, MAX_LEN_TTP, batch_size=32)

        sims_sub = cosine_sim_matrix(sent_embs, sub_embs)
        sent_i, sub_j = np.unravel_index(np.argmax(sims_sub), sims_sub.shape)

        best_score = float(sims_sub[sent_i, sub_j])
        best_sentence = sentences[sent_i]
        best_sub = subs[sub_j]

        doc["TTP_sentences"][parent] = [
            {
                "sentence": best_sentence,
                "score": round(best_score, 4),
                "matched_subtechnique": best_sub,
                "fallback": True,
            }
        ]

    return doc


# -------------------------
# GLOBAL BEST PER TTP (NEW)
# -------------------------
def update_global_best(
    global_best: Dict[str, Dict[str, Any]],
    doc: dict,
    source_file: str,
) -> None:
    """
    For each TTP in this doc, pick the highest-scoring sentence and update a global best map.
    """
    ttp_map = doc.get("TTP_sentences", {}) or {}
    for ttp_id, matches in ttp_map.items():
        if not matches:
            continue

        best = max(matches, key=lambda x: float(x.get("score", -1.0)))
        score = float(best.get("score", -1.0))

        current = global_best.get(ttp_id)
        if (current is None) or (score > float(current.get("score", -1.0))):
            global_best[ttp_id] = {
                "sentence": best.get("sentence", ""),
                "score": score,
                "source_file": source_file,
                "original_folder": str(INPUT_DIR),
            }


def iter_json_files(root: Path, recursive: bool) -> List[Path]:
    return sorted(root.rglob("*.json")) if recursive else sorted(root.glob("*.json"))


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    model = AutoModel.from_pretrained(MODEL_DIR).to(DEVICE)

    with open(TTP_DESC_PATH, "r", encoding="utf-8") as f:
        ttp_desc = json.load(f)

    files = iter_json_files(INPUT_DIR, RECURSIVE)
    if not files:
        print(f"⚠️ No .json files found in: {INPUT_DIR}")
        return

    global_best: Dict[str, Dict[str, Any]] = {}

    ok, failed = 0, 0
    for in_path in files:
        try:
            with open(in_path, "r", encoding="utf-8") as f:
                doc = json.load(f)

            doc = process_doc(doc, model, tokenizer, ttp_desc)

            # NEW: update global best-per-TTP using this doc
            update_global_best(global_best, doc, source_file=str(in_path))
            out_path = OUTPUT_DIR / in_path.name

            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(doc, f, indent=2, ensure_ascii=False)

            ok += 1
            print(f"✅ {in_path.name} -> {out_path}")

        except Exception as e:
            failed += 1
            print(f"❌ Failed on {in_path}: {e}")

    # NEW: write global best-per-TTP file
    best_path = OUTPUT_DIR / "best_sentence_per_ttp.json"
    with open(best_path, "w", encoding="utf-8") as f:
        json.dump(global_best, f, indent=2, ensure_ascii=False)

    print(f"\nDone. Success: {ok} | Failed: {failed}")
    print(f"Per-doc outputs: {OUTPUT_DIR}")
    print(f"Global best-per-TTP: {best_path}")


if __name__ == "__main__":
    main()


✅ row_00000.attack.json -> /home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/row_00000.attack.json
✅ row_00001.attack.json -> /home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/row_00001.attack.json
✅ row_00002.attack.json -> /home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/row_00002.attack.json
✅ row_00003.attack.json -> /home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/row_00003.attack.json
✅ row_00004.attack.json -> /home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/row_00004.attack.json
✅ row_00005.attack.json -> /home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/row_00005.attack.json
✅ row_00006.attack.json -> /home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/row_00006.attack.json
✅ row_00007.attack.json -> /home/simonettos/thijs/data_augment

Create augmentation file

In [122]:
#!/usr/bin/env python3
import json
from pathlib import Path
from typing import Dict, Any, List

# -----------------------------
# CONFIG
# -----------------------------
INPUT_FILES = [
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mitre_reports/mitre_rep_with_ttps/best_sentence_per_ttp.json",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mapedia_with_ttps/best_sentence_per_ttp.json",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_eset_with_ttps/best_sentence_per_ttp.json",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_aptnotes_with_ttps/best_sentence_per_ttp.json",
    "/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_apt_cybercriminals_with_ttps/best_sentence_per_ttp.json",
    "/home/simonettos/thijs/data_augmentatio_stefano/rcatt/files_with_ids_rcatt_with_ttps/best_sentence_per_ttp.json"
]
OUTPUT_FILE = "/home/simonettos/thijs/data_augmentatio_stefano/combined_6th_bosch.json"

# If True: merge rows by identical sentence text (recommended)
DEDUP_BY_SENTENCE = True


# -----------------------------
# Helpers
# -----------------------------
def make_doc_title(source_file: str) -> str:
    """
    Extract a readable title from filenames like:
      T1003.004__passcape_lsa_secrets__11ae09f367__436.attack.json
    -> "passcape lsa secrets"
    Fallbacks gracefully if naming differs.
    """
    name = Path(source_file).name
    parts = name.split("__")
    if len(parts) >= 2:
        title_part = parts[1]
    else:
        title_part = Path(source_file).stem
    title_part = title_part.replace("_", " ").strip()
    return title_part


def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# -----------------------------
# Main
# -----------------------------
def main() -> None:
    rows: List[Dict[str, Any]] = []
    sentence_index: Dict[str, int] = {}  # sentence -> row_idx (in rows)

    for fp in INPUT_FILES:
        data = load_json(fp)

        # data shape: { "Txxxx": {"sentence":..., "score":..., "source_file":..., ...}, ... }
        for ttp, obj in data.items():
            sent = (obj.get("sentence") or "").strip()
            if not sent:
                continue

            score = obj.get("score", None)
            source_file = obj.get("source_file", "")
            doc_title = make_doc_title(source_file)

            if DEDUP_BY_SENTENCE:
                if sent in sentence_index:
                    idx = sentence_index[sent]
                    # merge labels
                    if ttp not in rows[idx]["labels"]:
                        rows[idx]["labels"].append(ttp)
                    # keep best score (optional)
                    if score is not None:
                        prev = rows[idx].get("best_score")
                        if prev is None or score > prev:
                            rows[idx]["best_score"] = score
                    # keep first doc_title by default; you could also overwrite/merge if you want
                else:
                    sentence_index[sent] = len(rows)
                    rows.append(
                        {
                            "sentence": sent,
                            "labels": [ttp],
                            "doc_title": doc_title,
                            "best_score": score,
                        }
                    )
            else:
                rows.append(
                    {
                        "sentence": sent,
                        "labels": [ttp],
                        "doc_title": doc_title,
                        "best_score": score,
                    }
                )

    # Build the exact structure you showed: columns -> {"col": {"0":..., "1":...}, ...}
    out = {"sentence": {}, "labels": {}, "doc_title": {}}

    for i, r in enumerate(rows):
        key = str(i)
        out["sentence"][key] = r["sentence"]
        out["labels"][key] = r["labels"]
        out["doc_title"][key] = r["doc_title"]

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(rows)} rows to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Wrote 2290 rows to: /home/simonettos/thijs/data_augmentatio_stefano/combined_6th_bosch.json


In [104]:
import json
import re
import hashlib
from collections import defaultdict
from pathlib import Path

def normalize(s: str) -> str:
    """
    Normalization for 'same sentence' detection.
    Tweak as needed:
      - lowercasing
      - collapse whitespace
      - strip leading/trailing space
      - optionally remove punctuation
    """
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def load_sentence_map(json_path: str | Path) -> dict[str, str]:
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    if "sentence" not in data or not isinstance(data["sentence"], dict):
        raise ValueError("Expected JSON with top-level key 'sentence' containing a dict of id->text.")
    # Ensure values are strings
    sent_map = {}
    for k, v in data["sentence"].items():
        if isinstance(v, str):
            sent_map[str(k)] = v
    return sent_map

def find_duplicates(sent_map: dict[str, str], use_normalization: bool = True, min_len: int = 1):
    """
    Returns:
      duplicates: dict[signature -> list[(id, original_sentence)]]
    """
    groups = defaultdict(list)

    for sid, s in sent_map.items():
        if not isinstance(s, str):
            continue
        if len(s.strip()) < min_len:
            continue

        key_text = normalize(s) if use_normalization else s
        # Use hash to keep keys short (optional)
        sig = hashlib.sha1(key_text.encode("utf-8")).hexdigest()
        groups[sig].append((sid, s))

    duplicates = {sig: items for sig, items in groups.items() if len(items) > 1}
    return duplicates

def print_duplicate_report(duplicates: dict[str, list[tuple[str, str]]], show_sentence: bool = True, max_groups: int | None = None):
    sigs = list(duplicates.keys())
    if max_groups is not None:
        sigs = sigs[:max_groups]

    print(f"Found {len(duplicates)} duplicate group(s).")
    for i, sig in enumerate(sigs, 1):
        items = duplicates[sig]
        ids = [sid for sid, _ in items]
        print(f"\nGroup {i}: {len(items)} duplicates | ids={ids}")
        if show_sentence:
            # Print the first sentence as representative
            print("Representative sentence:")
            print(items[0][1])


json_path = "/home/simonettos/thijs/data_augmentatio_stefano/combined_6th_tram.json"
sent_map = load_sentence_map(json_path)

# Exact duplicates (character-for-character):
dup_exact = find_duplicates(sent_map, use_normalization=False)
print_duplicate_report(dup_exact)

# Normalized duplicates (case/whitespace-insensitive):
dup_norm = find_duplicates(sent_map, use_normalization=True)
print_duplicate_report(dup_norm)


Found 0 duplicate group(s).
Found 0 duplicate group(s).
